# Библтотеки

In [ ]:
!pip install pandas ujson

In [ ]:
import pandas as pd
import ujson as json
from pathlib import Path

In [ ]:
import json
import random
import re
from typing import Dict, List, Any, Tuple, Optional

In [ ]:
from dataclasses import dataclass
from enum import Enum

In [ ]:
import numpy as np

In [ ]:
from collections import defaultdict
import random

In [ ]:
!pip install spacy -q
!python -m spacy download en_core_web_sm -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 47.1 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
import spacy
nlp = spacy.load("en_core_web_sm")

In [ ]:
from datetime import datetime

# Загрузка датасета

In [ ]:
!git clone https://github.com/lorafei/Explainable_GEC.git
%cd Explainable_GEC
!ls

Cloning into 'Explainable_GEC'...
remote: Enumerating objects: 65, done.
remote: Counting objects: 100% (65/65), done.
remote: Compressing objects: 100% (49/49), done.
remote: Total 65 (delta 14), reused 65 (delta 14), pack-reused 0 (from 0)
Receiving objects: 100% (65/65), 5.44 MiB | 7.52 MiB/s, done.
Resolving deltas: 100% (14/14), done.
/content/Explainable_GEC/Explainable_GEC/Explainable_GEC/Explainable_GEC
cfgs  pic	     README.md	      run.py		  utils
data  read_jsonl.py  requirement.txt  simpletransformers


In [ ]:
data_path = Path("/content/Explainable_GEC/data/json/train.json")

rows = []
with data_path.open("r", encoding="utf-8") as f:
    for line in f:
        rows.append(json.loads(line))

In [ ]:
df = pd.DataFrame(rows)

In [ ]:
df.head()

,target,source,evidence_index,correction_index,error_type,predicted_parsing_order,origin
0,"[It, has, a, high, -, density, population, bec...","[It, has, a, high, -, density, population, bec...","[7, 9, 10, 11, 21, 23, 24, 25]","[8, 22]",Preposition,"{'1': 3, '5': 2, '7': 2, '8': 1, '9': 3, '10':...",A
1,"[Although, [NONE], it, is, an, industrial, cit...","[Although, of, it, is, an, industrial, city, ,...","[0, 17]","[1, 18]",Preposition,"{'1': 1, '18': 1}",A
2,"[Although, it, is, an, industrial, city, ,, th...","[Despite, it, is, an, industrial, city, ,, the...",[],"[0, 16]",Collocation,"{'0': 1, '5': 2, '8': 3, '16': 1, '21': 2, '24...",A
3,"[There, is, a, commercial, zone, along, the, w...","[There, are, a, commercial, zone, along, the, ...","[2, 3, 4, 50, 51, 52]","[1, 49]",Subject-Verb Agreement,"{'0': 2, '1': 1, '2': 3, '3': 2, '4': 3, '20':...",A
4,"[There, is, a, commercial, zone, along, the, w...","[There, are, a, commercial, zone, along, the, ...","[2, 3, 4, 50, 51, 52]","[1, 49]",Subject-Verb Agreement,"{'0': 2, '1': 1, '2': 3, '3': 2, '4': 3, '20':...",A


# Предобработка данных

In [ ]:
def detokenize(tokens) -> str:
    """
    Конвертирует токенизированные данные в обычный текст.
    Работает и со строкой, и со списком.

    Args:
        tokens: строка "It,has,a,..." ИЛИ список ["It", "has", "a", ...]

    Returns:
        Строка с пробелами и корректной пунктуацией
    """
    if isinstance(tokens, list):
        token_list = tokens
    elif isinstance(tokens, str):
        token_list = tokens.split(',')
    else:
        return str(tokens)

    no_space_before = {'.', ',', ':', ';', '!', '?', ')', ']', '}', "n't"}

    result = []
    for i, token in enumerate(token_list):
        token = str(token).strip()

        if token == '[NONE]':
            continue

        if i == 0:
            result.append(token)
        elif token in no_space_before:
            result.append(token)
        elif token.startswith("'") and i > 0:
            result.append(token)
        else:
            result.append(' ' + token)

    return ''.join(result)

In [ ]:
def load_dataset(data_path: str) -> pd.DataFrame:
    """Загружает JSONL датасет в DataFrame"""

    rows = []
    with open(data_path, 'r', encoding='utf-8') as f:
        for line in f:
            rows.append(json.loads(line))

    df = pd.DataFrame(rows)

    def parse_index_field(x):
        if isinstance(x, str):
            return [int(i) for i in x.split(',')] if x else []
        elif isinstance(x, list):
            return [int(i) for i in x] if x else []
        return []

    df['evidence_index'] = df['evidence_index'].apply(parse_index_field)
    df['correction_index'] = df['correction_index'].apply(parse_index_field)

    def parse_dict_field(x):
        if isinstance(x, str):
            return json.loads(x) if x else {}
        elif isinstance(x, dict):
            return x
        return {}

    df['predicted_parsing_order'] = df['predicted_parsing_order'].apply(parse_dict_field)

    return df

In [ ]:
df['source_text'] = df['source'].apply(detokenize)
df['target_text'] = df['target'].apply(detokenize)

for idx, row in df.head(5).iterrows():
    print(f"\n{'='*60}")
    print(f"ID: {idx} | Error Type: {row['error_type']}")
    print(f"{'='*60}")
    print(f"Source: {row['source_text']}")
    print(f"Target: {row['target_text']}")
    print(f"Error indices: {row['evidence_index']}")
    print(f"Correction indices: {row['correction_index']}")


ID: 0 | Error Type: Preposition
Source: It has a high - density population because its small territory.
Target: It has a high - density population because of its small territory.
Error indices: [7, 9, 10, 11, 21, 23, 24, 25]
Correction indices: [8, 22]

ID: 1 | Error Type: Preposition
Source: Although of it is an industrial city, there are many shops and department stores.
Target: Although it is an industrial city, there are many shops and department stores.
Error indices: [0, 17]
Correction indices: [1, 18]

ID: 2 | Error Type: Collocation
Source: Despite it is an industrial city, there are many shops and department stores.
Target: Although it is an industrial city, there are many shops and department stores.
Error indices: []
Correction indices: [0, 16]

ID: 3 | Error Type: Subject-Verb Agreement
Source: There are a commercial zone along the widest street of the city where you can find all kinds of businesses: banks, bars, chemists, cinemas, pet shops, restaurants, fast food restaur

# Fill-in-the-blanks

##  Извлечение слов для пропусков

In [ ]:
def analyze_correction_type(row: pd.Series) -> str:
    """
    Определяет тип исправления: insertion, deletion, или replacement
    """
    source_tokens = row['source'].split(',') if isinstance(row['source'], str) else row['source']
    target_tokens = row['target'].split(',') if isinstance(row['target'], str) else row['target']
    correction_idx = row['correction_index']

    source_clean = [t for t in source_tokens if t != '[NONE]']

    if len(target_tokens) > len(source_clean):
        return "insertion"
    elif len(target_tokens) < len(source_clean):
        return "deletion"
    else:
        return "replacement"

In [ ]:
df['correction_type'] = df.apply(analyze_correction_type, axis=1)
print("Распределение типов исправлений:")
print(df['correction_type'].value_counts())

Распределение типов исправлений:
correction_type
replacement    11064
insertion       3470
deletion         653
Name: count, dtype: int64


In [ ]:
def extract_blank_word(row: pd.Series) -> Optional[Dict]:
    """
    Извлекает слово/фразу для создания пропуска в упражнении.

    Returns:
        Dict с информацией о пропуске или None если не подходит
    """
    correction_type = row['correction_type']
    correction_idx = row['correction_index']

    source_tokens = row['source'].split(',') if isinstance(row['source'], str) else row['source']
    target_tokens = row['target'].split(',') if isinstance(row['target'], str) else row['target']

    source_clean = [t for t in source_tokens if t != '[NONE]']
    target_clean = [t for t in target_tokens if t != '[NONE]']

    result = {
        'suitable': True,
        'correction_type': correction_type,
        'blank_word': None,
        'wrong_answer': None,
        'blank_position': None,
    }

    # REPLACEMENT: слово заменено (лучший случай для FiB)
    if correction_type == 'replacement':
        if len(correction_idx) > 0:
            pos = correction_idx[0]

            if pos < len(target_clean):
                result['blank_word'] = target_clean[pos]
                result['blank_position'] = pos

            if len(row['evidence_index']) > 0:
                evidence_pos = row['evidence_index'][0]
                if evidence_pos < len(source_clean):
                    result['wrong_answer'] = source_clean[evidence_pos]

    # INSERTION: слово добавлено (хорошо для FiB)
    elif correction_type == 'insertion':
        if len(correction_idx) > 0:
            pos = correction_idx[0]

            if pos < len(target_clean):
                result['blank_word'] = target_clean[pos]
                result['blank_position'] = pos
                result['wrong_answer'] = ''

    # DELETION: слово удалено (проблематично для FiB)
    elif correction_type == 'deletion':
        result['suitable'] = False
        result['reason'] = 'Слово удалено из target, нельзя создать пропуск'

    return result

In [ ]:
blank_info = df.apply(extract_blank_word, axis=1)
df['blank_info'] = blank_info

df['blank_word'] = blank_info.apply(lambda x: x['blank_word'] if x else None)
df['blank_position'] = blank_info.apply(lambda x: x['blank_position'] if x else None)
df['wrong_answer'] = blank_info.apply(lambda x: x['wrong_answer'] if x else None)
df['is_suitable'] = blank_info.apply(lambda x: x['suitable'] if x else False)

In [ ]:
print("ПРИМЕРЫ ПОДХОДЯЩИХ УПРАЖНЕНИЙ")

replacement_examples = df[(df['correction_type'] == 'replacement') & (df['is_suitable'] == True)].head(3)
for idx, row in replacement_examples.iterrows():
    print(f"\n- ID: {idx} | Тип: {row['correction_type']} | Ошибка: {row['error_type']}")
    print(f"    Target: {row['target_text']}")
    print(f"    Пропуск: позиция {row['blank_position']}, слово: '{row['blank_word']}'")
    print(f"    Wrong answer: '{row['wrong_answer']}'")

insertion_examples = df[(df['correction_type'] == 'insertion') & (df['is_suitable'] == True)].head(2)
for idx, row in insertion_examples.iterrows():
    print(f"\n- ID: {idx} | Тип: {row['correction_type']} | Ошибка: {row['error_type']}")
    print(f"    Target: {row['target_text']}")
    print(f"    Пропуск: позиция {row['blank_position']}, слово: '{row['blank_word']}'")
    print(f"    Wrong answer: '(пусто - слово отсутствовало)'")

ПРИМЕРЫ ПОДХОДЯЩИХ УПРАЖНЕНИЙ

- ID: 1 | Тип: replacement | Ошибка: Preposition
    Target: Although it is an industrial city, there are many shops and department stores.
    Пропуск: позиция 1.0, слово: 'it'
    Wrong answer: 'Although'

- ID: 2 | Тип: replacement | Ошибка: Collocation
    Target: Although it is an industrial city, there are many shops and department stores.
    Пропуск: позиция 0.0, слово: 'Although'
    Wrong answer: 'None'

- ID: 3 | Тип: replacement | Ошибка: Subject-Verb Agreement
    Target: There is a commercial zone along the widest street of the city where you can find all kinds of businesses: banks, bars, chemists, cinemas, pet shops, restaurants, fast food restaurants, grocers, travel agencies, supermarkets and others.
    Пропуск: позиция 1.0, слово: 'is'
    Wrong answer: 'a'

- ID: 0 | Тип: insertion | Ошибка: Preposition
    Target: It has a high - density population because of its small territory.
    Пропуск: позиция 8.0, слово: 'of'
    Wrong answer:

## Генерация шаблонов с пропусками

In [ ]:
def create_blank_template(row: pd.Series) -> Optional[str]:
    """
    Создаёт текст упражнения с пропуском (_____) на месте целевого слова.
    """
    if not row.get('is_suitable', False):
        return None

    blank_pos = row.get('blank_position', None)

    if blank_pos is None or (isinstance(blank_pos, float) and np.isnan(blank_pos)):
        return None

    blank_pos = int(blank_pos)

    target_tokens = row['target'].split(',') if isinstance(row['target'], str) else row['target']
    target_clean = [t for t in target_tokens if t != '[NONE]']

    if blank_pos < 0 or blank_pos >= len(target_clean):
        return None

    if row.get('blank_word', None) is None:
        return None

    template_tokens = target_clean.copy()
    template_tokens[blank_pos] = '_____'

    result = []
    for i, token in enumerate(template_tokens):
        token = str(token).strip()
        if i == 0:
            result.append(token)
        elif token in {'.', ',', ':', ';', '!', '?', ')', ']', '}', "n't"}:
            result.append(token)
        else:
            result.append(' ' + token)

    return ''.join(result)

In [ ]:
df['exercise_template'] = df.apply(create_blank_template, axis=1)

In [ ]:
def build_exercise(row: pd.Series) -> Optional[Dict]:
    """
    Собирает полное упражнение в структурированном формате.
    """
    if not row['is_suitable'] or row['exercise_template'] is None:
        return None

    correct_answer = row['blank_word']
    wrong_answer = row['wrong_answer']
    error_type = row['error_type']

    distractors = []

    if wrong_answer and wrong_answer not in ['[NONE]', 'None', '']:
        distractors.append(wrong_answer)

    common_distractors = {
        'Preposition': ['in', 'on', 'at', 'by', 'for', 'with'],
        'Subject-Verb Agreement': ['is', 'are', 'was', 'were', 'has', 'have'],
        'Collocation': ['make', 'do', 'take', 'get', 'have'],
        'Article': ['a', 'an', 'the', ''],
    }

    if error_type in common_distractors:
        for d in common_distractors[error_type]:
            if d not in distractors and d != correct_answer and len(distractors) < 3:
                distractors.append(d)

    if len(distractors) < 3:
        target_tokens = row['target'].split(',') if isinstance(row['target'], str) else row['target']
        for t in target_tokens:
            if t not in distractors + [correct_answer] and t not in ['[NONE]', '.', ',', ':']:
                distractors.append(t)
                if len(distractors) >= 3:
                    break

    return {
        'exercise_id': row.name,
        'template': row['exercise_template'],
        'correct_answer': correct_answer,
        'options': [correct_answer] + distractors[:3],
        'error_type': error_type,
        'correction_type': row['correction_type'],
    }

In [ ]:
exercises = df.apply(build_exercise, axis=1)
exercises = exercises.dropna().tolist()

In [ ]:
print("ПРИМЕРЫ СГЕНЕРИРОВАННЫХ УПРАЖНЕНИЙ (Fill-in-the-Blanks)")

for i, ex in enumerate(exercises[:5]):
    print(f"\n{'='*60}")
    print(f"Exercise ID: {ex['exercise_id']}")
    print(f"Error Type: {ex['error_type']} | Correction: {ex['correction_type']}")
    print(f"{'='*60}")
    print(f"{ex['template']}")
    print(f"\n- Варианты ответов:")
    for j, opt in enumerate(ex['options']):
        marker = "+" if opt == ex['correct_answer'] else "  "
        print(f"   {marker} {j+1}. {opt}")

ПРИМЕРЫ СГЕНЕРИРОВАННЫХ УПРАЖНЕНИЙ (Fill-in-the-Blanks)

Exercise ID: 0
Error Type: Preposition | Correction: insertion
It has a high - density population because _____ its small territory.

- Варианты ответов:
   + 1. of
      2. in
      3. on
      4. at

Exercise ID: 1
Error Type: Preposition | Correction: replacement
Although _____ is an industrial city, there are many shops and department stores.

- Варианты ответов:
   + 1. it
      2. Although
      3. in
      4. on

Exercise ID: 2
Error Type: Collocation | Correction: replacement
_____ it is an industrial city, there are many shops and department stores.

- Варианты ответов:
   + 1. Although
      2. make
      3. do
      4. take

Exercise ID: 3
Error Type: Subject-Verb Agreement | Correction: replacement
There _____ a commercial zone along the widest street of the city where you can find all kinds of businesses: banks, bars, chemists, cinemas, pet shops, restaurants, fast food restaurants, grocers, travel agencies, supermar

## Улучшение с spaCy POS-теггером

In [ ]:
def get_pos_spacy(word: str) -> str:
    """
    Определяет часть речи слова через spaCy.
    Возвращает упрощённую категорию.
    """
    if not word or word in ['[NONE]', 'None', '', None]:
        return 'OTHER'

    try:
        doc = nlp(word)
        if len(doc) == 0:
            return 'OTHER'

        pos_tag = doc[0].pos_

        pos_mapping = {
            'ADP': 'PREP',      # Предлоги (in, on, at...)
            'DET': 'ART',       # Артикли и детерминаторы (a, an, the)
            'PRON': 'PRON',     # Местоимения (it, they, he...)
            'AUX': 'AUX',       # Вспомогательные глаголы (is, are, has...)
            'VERB': 'VERB',     # Глаголы (make, do, take...)
            'NOUN': 'NOUN',     # Существительные
            'PROPN': 'NOUN',    # Имена собственные → тоже NOUN
            'ADJ': 'ADJ',       # Прилагательные
            'ADV': 'ADV',       # Наречия
            'CCONJ': 'CONJ',    # Союзы (and, but, or)
            'SCONJ': 'CONJ',    # Подчинительные союзы (although, because)
            'PART': 'PART',     # Частицы (to в инфинитиве)
            'NUM': 'NUM',       # Числа
        }

        return pos_mapping.get(pos_tag, 'OTHER')

    except Exception as e:
        return 'OTHER'

In [ ]:
def build_spacy_pool(df: pd.DataFrame) -> Dict[str, Dict[str, List[str]]]:
    """
    Строит пул дистракторов с POS-категориями от spaCy.
    """
    pool = defaultdict(lambda: defaultdict(list))

    for idx, row in df.iterrows():
        error_type = row['error_type']
        wrong_answer = row.get('wrong_answer', None)

        if not wrong_answer or wrong_answer in ['[NONE]', 'None', '', None]:
            continue

        pos = get_pos_spacy(wrong_answer)

        if wrong_answer not in pool[error_type][pos]:
            pool[error_type][pos].append(wrong_answer)

        if (idx + 1) % 3000 == 0:
            print(f"   Обработано {idx + 1}/{len(df)} записей...")

    base_by_pos = {
        'PREP': ['in', 'on', 'at', 'by', 'for', 'with', 'from', 'to', 'of', 'about', 'into', 'upon'],
        'ART': ['a', 'an', 'the'],
        'PRON': ['it', 'they', 'them', 'he', 'she', 'we', 'you', 'this', 'that'],
        'AUX': ['is', 'are', 'was', 'were', 'has', 'have', 'had', 'does', 'do', 'did', 'will', 'would'],
        'CONJ': ['although', 'because', 'however', 'while', 'since', 'if', 'when'],
        'VERB': ['make', 'do', 'take', 'get', 'give', 'find', 'see', 'know', 'think'],
        'NOUN': ['time', 'way', 'day', 'man', 'thing', 'world', 'life', 'hand'],
    }

    for error_type in pool.keys():
        for pos_cat, words in base_by_pos.items():
            for w in words:
                if w not in pool[error_type][pos_cat]:
                    pool[error_type][pos_cat].append(w)

    return {k: dict(v) for k, v in pool.items()}

In [ ]:
def generate_spacy_distractors(row: pd.Series, pool: Dict, n_distractors: int = 3) -> List[str]:
    """Генерирует дистракторы той же POS категории что и правильный ответ."""

    correct_answer = row.get('blank_word', None)
    error_type = row.get('error_type', 'General')
    wrong_answer = row.get('wrong_answer', None)

    if not correct_answer:
        return []

    correct_pos = get_pos_spacy(correct_answer)

    distractors = set()

    if wrong_answer and wrong_answer not in ['[NONE]', 'None', '', None]:
        wrong_pos = get_pos_spacy(wrong_answer)
        if wrong_pos == correct_pos and wrong_answer != correct_answer:
            distractors.add(wrong_answer)

    if error_type in pool and correct_pos in pool[error_type]:
        candidate_pool = [w for w in pool[error_type][correct_pos] if w != correct_answer]

        n_needed = n_distractors - len(distractors)
        if len(candidate_pool) >= n_needed:
            sampled = random.sample(candidate_pool, n_needed)
            distractors.update(sampled)
        else:
            distractors.update(candidate_pool)

    if len(distractors) < n_distractors and error_type in pool:
        for pos_cat, words in pool[error_type].items():
            if pos_cat != correct_pos:
                for w in words:
                    if w != correct_answer and w not in distractors:
                        distractors.add(w)
                        if len(distractors) >= n_distractors:
                            break
            if len(distractors) >= n_distractors:
                break

    distractors_list = list(distractors)[:n_distractors]
    random.shuffle(distractors_list)

    return distractors_list

In [ ]:
def build_exercise_spacy(row: pd.Series, pool: Dict) -> Optional[Dict]:
    """Собирает упражнение с spaCy POS-фильтрованными дистракторами."""

    if not row.get('is_suitable', False) or row.get('exercise_template') is None:
        return None

    correct_answer = row['blank_word']
    error_type = row['error_type']
    correct_pos = get_pos_spacy(correct_answer)

    distractors = generate_spacy_distractors(row, pool, n_distractors=3)

    options = [correct_answer] + distractors
    random.shuffle(options)

    return {
        'exercise_id': int(row.name),
        'template': row['template'] if 'template' in row else row['exercise_template'],
        'correct_answer': correct_answer,
        'options': options,
        'error_type': error_type,
        'correction_type': row['correction_type'],
        'correct_pos': correct_pos,
    }

In [ ]:
spacy_pool = build_spacy_pool(df)

   Обработано 3000/15187 записей...
   Обработано 6000/15187 записей...
   Обработано 15000/15187 записей...


In [ ]:
exercises_spacy = df.apply(lambda row: build_exercise_spacy(row, spacy_pool), axis=1)
exercises_spacy = exercises_spacy.dropna().tolist()

print(f"\nВсего сгенерировано упражнений: {len(exercises_spacy)}")
print(f"Из {len(df)} записей датасета ({len(exercises_spacy)/len(df)*100:.1f}%)")


Всего сгенерировано упражнений: 14533
Из 15187 записей датасета (95.7%)


In [ ]:
print("ПРИМЕРЫ ФИНАЛЬНЫХ УПРАЖНЕНИЙ")

for ex in exercises_spacy[:5]:
    print(f"\n{'─'*60}")
    print(f"ID: {ex['exercise_id']}  |  Тип ошибки: {ex['error_type']}")
    print(f"{'─'*60}")
    print(f"{ex['template']}")
    print(f"\nВарианты ответов:")
    for j, opt in enumerate(ex['options']):
        marker = "✓" if opt == ex['correct_answer'] else " "
        opt_pos = get_pos_spacy(opt)
        print(f"   [{marker}] {j+1}. {opt}  ")
    print(f"\nПравильный ответ: {ex['correct_answer']}")

ПРИМЕРЫ ФИНАЛЬНЫХ УПРАЖНЕНИЙ

────────────────────────────────────────────────────────────
ID: 0  |  Тип ошибки: Preposition
────────────────────────────────────────────────────────────
It has a high - density population because _____ its small territory.

Варианты ответов:
   [ ] 1. in  
   [ ] 2. to  
   [✓] 3. of  
   [ ] 4. From  

Правильный ответ: of

────────────────────────────────────────────────────────────
ID: 1  |  Тип ошибки: Preposition
────────────────────────────────────────────────────────────
Although _____ is an industrial city, there are many shops and department stores.

Варианты ответов:
   [ ] 1. everything  
   [ ] 2. any  
   [✓] 3. it  
   [ ] 4. this  

Правильный ответ: it

────────────────────────────────────────────────────────────
ID: 2  |  Тип ошибки: Collocation
────────────────────────────────────────────────────────────
_____ it is an industrial city, there are many shops and department stores.

Варианты ответов:
   [ ] 1. because  
   [ ] 2. if  
   

## Экспорт

In [ ]:
def export_to_platform_format(exercises: List[Dict], output_path: str):
    """
    Сохраняет упражнения в формате для платформы SAYIT.
    """
    export_data = {
        "metadata": {
            "dataset": "Explainable_GEC",
            "method": "template-based",
            "generation_date": datetime.now().isoformat(),
            "total_exercises": len(exercises),
            "version": "1.0"
        },
        "exercises": []
    }

    for ex in exercises:
        exercise_record = {
            "id": ex['exercise_id'],
            "type": "fill-in-the-blanks",
            "error_type": ex['error_type'],
            "correction_type": ex['correction_type'],
            "question": ex['template'],
            "correct_answer": ex['correct_answer'],
            "correct_pos": ex['correct_pos'],
            "options": ex['options'],
            "difficulty": "auto",
            "explanation": ""
        }
        export_data["exercises"].append(exercise_record)

    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(export_data, f, ensure_ascii=False, indent=2)

    print(f"Сохранено {len(exercises)} упражнений в {output_path}")
    return export_data

In [ ]:
def generate_statistics(exercises: List[Dict], df: pd.DataFrame) -> Dict:
    """
    Генерирует статистику для включения в ВКР.
    """
    stats = {
        "dataset": {
            "total_records": len(df),
            "suitable_for_fib": int(df['is_suitable'].sum()),
            "success_rate": f"{len(exercises)/len(df)*100:.1f}%"
        },
        "error_types": {},
        "pos_distribution": {},
        "correction_types": {}
    }

    for ex in exercises:
        et = ex['error_type']
        stats['error_types'][et] = stats['error_types'].get(et, 0) + 1

    for ex in exercises:
        pos = ex['correct_pos']
        stats['pos_distribution'][pos] = stats['pos_distribution'].get(pos, 0) + 1

    for ex in exercises:
        ct = ex['correction_type']
        stats['correction_types'][ct] = stats['correction_types'].get(ct, 0) + 1

    return stats

In [ ]:
export_data = export_to_platform_format(exercises_spacy, "sayit_exercises_template.json")

Сохранено 14533 упражнений в sayit_exercises_template.json


In [ ]:
# Статистика
stats = generate_statistics(exercises_spacy, df)

print(f"\nДатасет:")
print(f"Всего записей: {stats['dataset']['total_records']}")
print(f"Пригодно для FiB: {stats['dataset']['suitable_for_fib']}")
print(f"Успешность генерации: {stats['dataset']['success_rate']}")

print(f"\nТипы ошибок (топ-5):")
sorted_errors = sorted(stats['error_types'].items(), key=lambda x: -x[1])[:5]
for error_type, count in sorted_errors:
    print(f"{error_type}: {count} ({count/len(exercises_spacy)*100:.1f}%)")

print(f"\nPOS категории правильных ответов:")
for pos, count in sorted(stats['pos_distribution'].items(), key=lambda x: -x[1]):
    print(f"{pos}: {count} ({count/len(exercises_spacy)*100:.1f}%)")

print(f"\nТипы исправлений:")
for ct, count in stats['correction_types'].items():
    print(f"{ct}: {count} ({count/len(exercises_spacy)*100:.1f}%)")

with open("sayit_statistics_template.json", 'w', encoding='utf-8') as f:
    json.dump(stats, f, ensure_ascii=False, indent=2)


Датасет:
Всего записей: 15187
Пригодно для FiB: 14534
Успешность генерации: 95.7%

Типы ошибок (топ-5):
Preposition: 2066 (14.2%)
Verb Tense: 1848 (12.7%)
Number: 1722 (11.8%)
Collocation: 1512 (10.4%)
Others: 1154 (7.9%)

POS категории правильных ответов:
VERB: 3873 (26.6%)
NOUN: 3188 (21.9%)
PREP: 1778 (12.2%)
PRON: 1553 (10.7%)
AUX: 1390 (9.6%)
PART: 845 (5.8%)
ADJ: 671 (4.6%)
ADV: 612 (4.2%)
CONJ: 358 (2.5%)
OTHER: 219 (1.5%)
NUM: 45 (0.3%)
ART: 1 (0.0%)

Типы исправлений:
insertion: 3470 (23.9%)
replacement: 11063 (76.1%)


# Реконструкция предложений

In [ ]:
def get_tokens_from_row(row: pd.Series, column: str = 'target') -> List[str]:
    """
    Извлекает токены из готовых данных датасета.

    Args:
        row: строка DataFrame
        column: 'target' или 'source'

    Returns:
        Список токенов без [NONE] и пустых значений
    """
    tokens_raw = row.get(column, '')

    if isinstance(tokens_raw, str):
        tokens = tokens_raw.split(',')
    elif isinstance(tokens_raw, list):
        tokens = tokens_raw
    else:
        return []

    tokens = [t.strip() for t in tokens if t.strip() and t != '[NONE]']

    return tokens

In [ ]:
def shuffle_tokens(tokens: List[str], difficulty: str = 'medium') -> List[str]:
    """
    Перемешивает токены с разным уровнем сложности.
    """
    if len(tokens) <= 2:
        return tokens.copy()

    shuffled = tokens.copy()

    if difficulty == 'easy':
        if len(shuffled) > 2:
            middle = shuffled[1:-1]
            random.shuffle(middle)
            shuffled = [shuffled[0]] + middle + [shuffled[-1]]

    elif difficulty == 'medium':
        random.shuffle(shuffled)

    elif difficulty == 'hard':
        random.shuffle(shuffled)

    return shuffled

In [ ]:
def is_suitable_for_reconstruction(row: pd.Series) -> bool:
    """
    Проверяет пригодность предложения для упражнения на реконструкцию.
    """
    tokens = get_tokens_from_row(row, 'target')

    if len(tokens) < 4 or len(tokens) > 20:
        return False

    return True

In [ ]:
def create_reconstruction_exercise(row: pd.Series, difficulty: str = 'medium') -> Optional[Dict]:
    """
    Создаёт упражнение на восстановление порядка слов.
    """
    if not is_suitable_for_reconstruction(row):
        return None

    tokens = get_tokens_from_row(row, 'target')
    target_text = row.get('target_text', '')
    source_text = row.get('source_text', '')

    shuffled = shuffle_tokens(tokens, difficulty)

    exercise = {
        'exercise_id': int(row.name),
        'type': 'sentence-reconstruction',
        'error_type': row.get('error_type', 'Unknown'),
        'correction_type': row.get('correction_type', 'Unknown'),
        'difficulty': difficulty,
        'shuffled_tokens': shuffled,
        'correct_order': tokens,
        'correct_sentence': target_text,
        'source_sentence': source_text,
        'num_tokens': len(tokens),
    }

    return exercise

In [ ]:
df['suitable_for_reconstruction'] = df.apply(is_suitable_for_reconstruction, axis=1)
suitable_count = df['suitable_for_reconstruction'].sum()

print(f"\nПригодных записей: {suitable_count} из {len(df)} ({suitable_count/len(df)*100:.1f}%)")


Пригодных записей: 6138 из 15187 (40.4%)


In [ ]:
reconstruction_exercises = []

for difficulty in ['easy', 'medium', 'hard']:
    print(f"\nГенерация (сложность: {difficulty})...")

    suitable_df = df[df['suitable_for_reconstruction'] == True]

    for idx, row in suitable_df.iterrows():
        exercise = create_reconstruction_exercise(row, difficulty)
        if exercise:
            reconstruction_exercises.append(exercise)

print(f"\nВсего сгенерировано упражнений: {len(reconstruction_exercises)}")


Генерация (сложность: easy)...

Генерация (сложность: medium)...

Генерация (сложность: hard)...

Всего сгенерировано упражнений: 18414


In [ ]:
print("ПРИМЕРЫ УПРАЖНЕНИЙ НА РЕКОНСТРУКЦИЮ")

for difficulty in ['easy', 'medium', 'hard']:
    examples = [ex for ex in reconstruction_exercises if ex['difficulty'] == difficulty][:1]

    for ex in examples:
        print(f"\n{'─'*60}")
        print(f"ID: {ex['exercise_id']}  |  Сложность: {ex['difficulty'].upper()}")
        print(f"Тип ошибки: {ex['error_type']}  |  Токенов: {ex['num_tokens']}")
        print(f"{'─'*60}")
        print(f"\nЗадание: Восстановите правильный порядок слов")
        print(f"\nПеремешанные слова:")
        print(f"   {' | '.join(ex['shuffled_tokens'])}")
        print(f"\nПравильный ответ:")
        print(f"   {ex['correct_sentence']}")

ПРИМЕРЫ УПРАЖНЕНИЙ НА РЕКОНСТРУКЦИЮ

────────────────────────────────────────────────────────────
ID: 0  |  Сложность: EASY
Тип ошибки: Preposition  |  Токенов: 13
────────────────────────────────────────────────────────────

Задание: Восстановите правильный порядок слов

Перемешанные слова:
   It | its | territory | a | - | because | high | small | has | population | density | of | .

Правильный ответ:
   It has a high - density population because of its small territory.

────────────────────────────────────────────────────────────
ID: 0  |  Сложность: MEDIUM
Тип ошибки: Preposition  |  Токенов: 13
────────────────────────────────────────────────────────────

Задание: Восстановите правильный порядок слов

Перемешанные слова:
   density | territory | has | population | of | a | small | . | because | high | - | its | It

Правильный ответ:
   It has a high - density population because of its small territory.

────────────────────────────────────────────────────────────
ID: 0  |  Сложност

## Экспорт

In [ ]:
def export_reconstruction_to_json(exercises: List[Dict], output_path: str):
    """
    Сохраняет упражнения на реконструкцию в формате для платформы SAYIT.
    """
    export_data = {
        "metadata": {
            "dataset": "Explainable_GEC",
            "method": "template-based",
            "exercise_type": "sentence-reconstruction",
            "generation_date": datetime.now().isoformat(),
            "total_exercises": len(exercises),
            "version": "1.0"
        },
        "exercises": []
    }

    for ex in exercises:
        exercise_record = {
            "id": ex['exercise_id'],
            "type": "sentence-reconstruction",
            "error_type": ex['error_type'],
            "correction_type": ex['correction_type'],
            "difficulty": ex['difficulty'],
            "question": "Восстановите правильный порядок слов",
            "shuffled_tokens": ex['shuffled_tokens'],
            "correct_order": ex['correct_order'],
            "correct_sentence": ex['correct_sentence'],
            "source_sentence": ex['source_sentence'],
            "num_tokens": ex['num_tokens'],
        }
        export_data["exercises"].append(exercise_record)

    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(export_data, f, ensure_ascii=False, indent=2)

    print(f"Сохранено {len(exercises)} упражнений в {output_path}")
    return export_data

In [ ]:
export_data = export_reconstruction_to_json(reconstruction_exercises, "exercises_reconstruction.json")

Сохранено 18414 упражнений в exercises_reconstruction.json


In [ ]:
def generate_reconstruction_statistics(exercises: List[Dict], df: pd.DataFrame) -> Dict:
    """
    Генерирует статистику для включения в ВКР.
    """
    stats = {
        "dataset": {
            "total_records": len(df),
            "suitable_for_reconstruction": int(df['suitable_for_reconstruction'].sum()),
            "success_rate": f"{len(exercises)/len(df)/3*100:.1f}%"  # Делим на 3 т.к. 3 уровня сложности
        },
        "difficulty_distribution": {},
        "error_types": {},
        "token_length_distribution": {}
    }

    for ex in exercises:
        diff = ex['difficulty']
        stats['difficulty_distribution'][diff] = stats['difficulty_distribution'].get(diff, 0) + 1

    for ex in exercises:
        et = ex['error_type']
        stats['error_types'][et] = stats['error_types'].get(et, 0) + 1

    for ex in exercises:
        num_tokens = ex['num_tokens']
        if num_tokens <= 6:
            length_cat = "short (4-6)"
        elif num_tokens <= 12:
            length_cat = "medium (7-12)"
        else:
            length_cat = "long (13-20)"

        stats['token_length_distribution'][length_cat] = stats['token_length_distribution'].get(length_cat, 0) + 1

    return stats

In [ ]:
stats_reconstruction = generate_reconstruction_statistics(reconstruction_exercises, df)

print(f"\nДатасет:")
print(f"Всего записей: {stats_reconstruction['dataset']['total_records']}")
print(f"Пригодно для реконструкции: {stats_reconstruction['dataset']['suitable_for_reconstruction']}")
print(f"Успешность генерации: {stats_reconstruction['dataset']['success_rate']}")

print(f"\nРаспределение по сложности:")
for diff, count in sorted(stats_reconstruction['difficulty_distribution'].items()):
    print(f"{diff}: {count} ({count/len(reconstruction_exercises)*100:.1f}%)")

print(f"\nТипы ошибок (топ-5):")
sorted_errors = sorted(stats_reconstruction['error_types'].items(), key=lambda x: -x[1])[:5]
for error_type, count in sorted_errors:
    print(f"{error_type}: {count} ({count/len(reconstruction_exercises)*100:.1f}%)")

print(f"\nДлина предложений:")
for length_cat, count in sorted(stats_reconstruction['token_length_distribution'].items(), key=lambda x: -x[1]):
    print(f"{length_cat}: {count} ({count/len(reconstruction_exercises)*100:.1f}%)")


Датасет:
   Всего записей: 15187
   Пригодно для реконструкции: 6138
   Успешность генерации: 40.4%

Распределение по сложности:
   easy: 6138 (33.3%)
   hard: 6138 (33.3%)
   medium: 6138 (33.3%)

Типы ошибок (топ-5):
   Verb Tense: 2604 (14.1%)
   Preposition: 2370 (12.9%)
   Number: 2196 (11.9%)
   Collocation: 1941 (10.5%)
   Others: 1539 (8.4%)

Длина предложений:
   long (13-20): 12879 (69.9%)
   medium (7-12): 5067 (27.5%)
   short (4-6): 468 (2.5%)
